# Slayer1B 1.09 Instruct — benchmark na GPU (Colab)

Dwa pomiary, oba na tym samym modelu, ale o różnym znaczeniu:

| co | jak | po co |
|---|---|---|
| **zestaw deterministyczny** (`scripts/bench_instruct.py`) | format / tool-call / pętle, sprawdzanie regexem i parsowaniem | czy model trzyma instrukcje i format — 3/6 i 3/6 na CPU |
| **panel lm-eval** (`slayer_kv_lm.py`) | gotowe zadania przez adapter Slayera, `--limit` na rozgrzewkę | liczby porównywalne z bazą 1.09 i panelem 50 zadań |

Czego panel **nie** zmierzy: adapter podaje prompt bez szablonu ChatML (raw continuation), więc
instruct wypada zaniżony, a zadania są w większości loglikelihood-owe (wiedza, nie posłuszeństwo).

**Wymagane:** Runtime → Change runtime type → **T4 GPU**. Model: 4,3 GB fp32.

Czas: paczka ~9 min (4,3 GB), nasz zestaw ~2–3 min, panel 2 zadań z `--limit 100` ~10–20 min.


## 1. Środowisko

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Włącz GPU: Runtime -> Change runtime type -> T4 GPU"


In [ ]:
%pip install -q tokenizers safetensors huggingface_hub psutil


## 2. Repo

In [ ]:
%%bash
rm -rf /content/machinery /content/bf
git clone -q -b feat/slayer1b-109-instruct https://github.com/PiotrStyla/Machinery_Instruction_Assistent.git /content/machinery
git clone -q --depth 1 https://github.com/slayerlabs/benchmarks-fabryka-ai.git /content/bf
cd /content/machinery && git log --oneline -2 && ls scripts | head


## 3. Paczka modelu (przypięte rewizje + loader dla harnessu)

In [ ]:
!cd machinery && python scripts/fetch_slayer_109_instruct.py --dry-run


In [ ]:
%%bash
cd /content/machinery
python scripts/fetch_slayer_109_instruct.py 2>&1 | tail -5


## 4. Zestaw deterministyczny na GPU (te same prompty co na CPU)

In [ ]:
%%bash
cd /content/machinery
python scripts/bench_instruct.py --model-dir models/Slayer1B-1.09-Instruct \
  --device cuda --only loops --loop-tokens 192 --out reports/gpu-loops.json 2>&1 | tail -8


In [ ]:
%%bash
cd /content/machinery
python scripts/bench_instruct.py --model-dir models/Slayer1B-1.09-Instruct \
  --device cuda --only loops --loop-tokens 192 --out reports/gpu-loops.json 2>&1 | tail -8


## 5. Panel lm-eval (adapter Slayera, raw continuation)

In [ ]:
%pip install -q lm-eval
import lm_eval; print('lm_eval', lm_eval.__version__)


In [ ]:
%%bash
cd /content/bf/research/common/slayer_serving
python slayer_kv_lm.py \
  --pkg /content/machinery/models/Slayer1B-1.09-Instruct \
  --tasks arc_challenge,piqa --limit 100 --batch 4 --device cuda \
  --out /content/machinery/reports/panel-1.09-instruct.json 2>&1 | tail -25


## 6. Podsumowanie

Kluczowe porównania po powrocie do repo:

- `bench_instruct` GPU vs CPU (`docs/slayer1b-109-instruct.md`): te same prompty, więc różni się tylko tok/s i ewentualnie wynik przy krótkich budżetach.
- panel vs baza 1.09: jeśli panel liczy się poprawnie, dostajesz `acc`/`acc_norm` dla arc_challenge i piqa.
- jeśli `slayer_kv_lm.py` wywali się na braku zadania albo na wersji `lm_eval` — zapisz błąd, to jest informacja o adapterze, nie o modelu.

In [ ]:
%%bash
cd /content/machinery
python scripts/bench_instruct.py --model-dir models/Slayer1B-1.09-Instruct \
  --device cuda --only loops --loop-tokens 192 --out reports/gpu-loops.json 2>&1 | tail -8


In [ ]:
!cd /content/machinery && zip -q reports-gpu.zip reports/*.json 2>/dev/null; ls -la reports/*.json
from google.colab import files
files.download("/content/machinery/reports-gpu.zip")
